# HyDE: reproduction on SciFact and ArguAna

The claim we test (Gao et al., ACL 2023, Table 2): searching with the embedding of LLM-written hypothetical documents beats searching with the query embedding, with the same unsupervised encoder (Contriever) and no training.

Paper, nDCG@10: SciFact 64.9 with Contriever and 69.1 with HyDE; ArguAna 37.9 and 46.6.

Same as the paper: `facebook/contriever`, mean pooling, dot product, full documents, the paper's prompts, N = 8, temperature 0.7, at most 512 tokens, the query vector included in the average.
This is the paper's protocol: full documents, dot product on raw embeddings. The testbed rows (`method.py`, `results_variants/`) use the first 200 words and cosine; the section "Same encoder, two protocols" compares the two.
Different: the generator. InstructGPT (`text-davinci-003`) is retired; we use Qwen2.5-3B-Instruct, 8-bit, on a laptop (`generate.py`).

Run from the repo root after `generate.py`, or with the committed `generations/`. Scores come from `testbed.score_run` (`pytrec_eval`; on ArguAna the query's own document is removed).

In [1]:
import sys
sys.path += ["../..", "."]
from sentence_transformers import SentenceTransformer
import testbed, hyde

model = SentenceTransformer("facebook/contriever", device=testbed.device())   # mean pooling
model.max_seq_length = 512

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: facebook/contriever
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Plain Contriever, then HyDE

For each dataset: the plain Contriever score (the check against the paper), then HyDE for N = 1, 2, 4, 8 hypothetical documents, with and without the query vector in the average. Every HyDE line is compared with plain Contriever on the same queries: mean difference, bootstrap 95 % CI, paired t-test.

In [2]:
kept = {}
for name in ["scifact", "arguana"]:
    corpus, queries, qrels = testbed.load(name)
    ids, qids = list(corpus), list(queries)
    docs = [(d["title"] + " " + d["text"]).strip() for d in corpus.values()]    # full text, no 200-word cut
    D = model.encode(docs, batch_size=32, convert_to_numpy=True)
    Q = model.encode(list(queries.values()), batch_size=32, convert_to_numpy=True)
    E = hyde.embed_generations(model, queries, hyde.load_generations(name))
    _, base, base_r = testbed.score_run(hyde.search(Q, D, qids, ids), name)
    kept[name] = (queries, ids, qids, D, Q, base, E)
    print(f"{name} Contriever            nDCG@10 {base.mean():.3f}  R@100 {base_r.mean():.3f}")
    for n in (1, 2, 4, 8):
        for label, q in (("with query", Q), ("docs only ", None)):
            _, nd, rc = testbed.score_run(hyde.search(hyde.hyde_vectors(E, q, n), D, qids, ids), name)
            c = testbed.compare(nd, base)
            print(f"{name} HyDE N={n} {label}  nDCG@10 {nd.mean():.3f}  R@100 {rc.mean():.3f}  "
                  f"delta {c['delta']:+.3f} {c['ci95']} p={c['p']:.3g}")

scifact Contriever            nDCG@10 0.649  R@100 0.926
scifact HyDE N=1 with query  nDCG@10 0.651  R@100 0.944  delta +0.001 [-0.0268, 0.0294] p=0.928


scifact HyDE N=1 docs only   nDCG@10 0.544  R@100 0.890  delta -0.105 [-0.1451, -0.0643] p=4.91e-07
scifact HyDE N=2 with query  nDCG@10 0.636  R@100 0.944  delta -0.013 [-0.0443, 0.0186] p=0.419


scifact HyDE N=2 docs only   nDCG@10 0.558  R@100 0.896  delta -0.092 [-0.1302, -0.053] p=4.88e-06
scifact HyDE N=4 with query  nDCG@10 0.617  R@100 0.930  delta -0.032 [-0.0655, 0.0001] p=0.0523


scifact HyDE N=4 docs only   nDCG@10 0.574  R@100 0.918  delta -0.076 [-0.1136, -0.0376] p=0.000117
scifact HyDE N=8 with query  nDCG@10 0.608  R@100 0.928  delta -0.042 [-0.0765, -0.0073] p=0.0193


scifact HyDE N=8 docs only   nDCG@10 0.584  R@100 0.918  delta -0.065 [-0.1028, -0.0275] p=0.000769


arguana Contriever            nDCG@10 0.379  R@100 0.901


... (4 outputs left out)

arguana HyDE N=4 with query  nDCG@10 0.223  R@100 0.818  delta -0.157 [-0.173, -0.1403] p=1.71e-69


arguana HyDE N=4 docs only   nDCG@10 0.142  R@100 0.690  delta -0.237 [-0.2561, -0.219] p=3.45e-115


arguana HyDE N=8 with query  nDCG@10 0.192  R@100 0.773  delta -0.187 [-0.2046, -0.1697] p=1.76e-84


arguana HyDE N=8 docs only   nDCG@10 0.147  R@100 0.692  delta -0.232 [-0.251, -0.2139] p=1.41e-110


## After the fact: why does HyDE lose here?

Everything below was run after we saw the scores above, so it is diagnosis, not the reproduction. The official result stays the one above.

Qwen answers like a chat assistant: headings (`### Passage: ...`), openings such as "Certainly! Here's a counter argument", bold marks and numbered lists, about 400 words per document, often cut off at the 512-token limit. Three checks on the same cached documents, N = 8 with the query vector, each against plain Contriever:

- cleaned: heading lines, chat openings and bold marks removed;
- first 100 words only;
- both.

In [3]:
import re

def clean(text):
    lines = [l for l in text.splitlines() if l.strip() and not re.match(r"\s*(#|\*\*Passage|Certainly|Counter Argument:\s*$)", l)]
    return re.sub(r"\*\*|^Counter Argument:\s*", "", " ".join(lines))

def first_words(text, k=100):
    return " ".join(text.split()[:k])

checks = {"cleaned": clean, "first 100 words": first_words, "cleaned, first 100 words": lambda t: first_words(clean(t))}
per_query = {}
for name, (queries, ids, qids, D, Q, base, _) in kept.items():
    generations = hyde.load_generations(name)
    for label, change in checks.items():
        changed = {q: [change(d) for d in docs] for q, docs in generations.items()}
        E = hyde.embed_generations(model, queries, changed)
        _, nd, rc = testbed.score_run(hyde.search(hyde.hyde_vectors(E, Q), D, qids, ids), name)
        per_query[name, label] = nd
        c = testbed.compare(nd, base)
        print(f"{name} HyDE {label:25} nDCG@10 {nd.mean():.3f}  R@100 {rc.mean():.3f}  "
              f"delta {c['delta']:+.3f} {c['ci95']} p={c['p']:.3g}")

scifact HyDE cleaned                   nDCG@10 0.607  R@100 0.929  delta -0.043 [-0.0765, -0.0085] p=0.0149


scifact HyDE first 100 words           nDCG@10 0.689  R@100 0.964  delta +0.040 [0.0114, 0.0696] p=0.00718


scifact HyDE cleaned, first 100 words  nDCG@10 0.690  R@100 0.954  delta +0.041 [0.0127, 0.0701] p=0.00546


arguana HyDE cleaned                   nDCG@10 0.207  R@100 0.790  delta -0.172 [-0.1898, -0.1556] p=9.23e-75


arguana HyDE first 100 words           nDCG@10 0.436  R@100 0.971  delta +0.057 [0.0373, 0.0764] p=1.69e-08


arguana HyDE cleaned, first 100 words  nDCG@10 0.450  R@100 0.973  delta +0.071 [0.0515, 0.09] p=9.89e-13


## Second run: plain passages

One regeneration with a changed setting (`python generate.py plain`, cache in `generations_plain/`): the same prompts, plus the system message `PLAIN` from `prompts.py` asking for one plain paragraph, and at most 256 tokens. Model, sampling and seed are unchanged. The setting was written before the three checks above were read and committed before this run started.

Same table as for the first run.

In [4]:
for name, (queries, ids, qids, D, Q, base, _) in kept.items():
    E = hyde.embed_generations(model, queries, hyde.load_generations(name, "generations_plain"))
    for n in (1, 2, 4, 8):
        for label, q in (("with query", Q), ("docs only ", None)):
            _, nd, rc = testbed.score_run(hyde.search(hyde.hyde_vectors(E, q, n), D, qids, ids), name)
            c = testbed.compare(nd, base)
            print(f"{name} plain N={n} {label}  nDCG@10 {nd.mean():.3f}  R@100 {rc.mean():.3f}  "
                  f"delta {c['delta']:+.3f} {c['ci95']} p={c['p']:.3g}")
    per_query[name, "second run"] = testbed.score_run(hyde.search(hyde.hyde_vectors(E, Q), D, qids, ids), name)[1]

scifact plain N=1 with query  nDCG@10 0.666  R@100 0.953  delta +0.016 [-0.0005, 0.034] p=0.0648
scifact plain N=1 docs only   nDCG@10 0.646  R@100 0.953  delta -0.004 [-0.031, 0.0245] p=0.804


scifact plain N=2 with query  nDCG@10 0.675  R@100 0.954  delta +0.026 [0.0076, 0.0442] p=0.00682
scifact plain N=2 docs only   nDCG@10 0.669  R@100 0.955  delta +0.020 [-0.0058, 0.0467] p=0.132


scifact plain N=4 with query  nDCG@10 0.679  R@100 0.966  delta +0.030 [0.011, 0.0491] p=0.00302
scifact plain N=4 docs only   nDCG@10 0.681  R@100 0.968  delta +0.031 [0.0082, 0.0555] p=0.0109


scifact plain N=8 with query  nDCG@10 0.682  R@100 0.968  delta +0.033 [0.0134, 0.0531] p=0.00173
scifact plain N=8 docs only   nDCG@10 0.688  R@100 0.964  delta +0.039 [0.0161, 0.063] p=0.00155


arguana plain N=1 with query  nDCG@10 0.453  R@100 0.973  delta +0.074 [0.0623, 0.0852] p=1.64e-34


arguana plain N=1 docs only   nDCG@10 0.401  R@100 0.948  delta +0.021 [-0.0004, 0.043] p=0.0523


... (2 outputs left out)

arguana plain N=4 with query  nDCG@10 0.484  R@100 0.982  delta +0.105 [0.0879, 0.1215] p=2.05e-32


arguana plain N=4 docs only   nDCG@10 0.443  R@100 0.967  delta +0.064 [0.0424, 0.0852] p=7.89e-09


arguana plain N=8 with query  nDCG@10 0.477  R@100 0.978  delta +0.098 [0.079, 0.1163] p=9.49e-24


arguana plain N=8 docs only   nDCG@10 0.451  R@100 0.969  delta +0.072 [0.0508, 0.0929] p=5.82e-11


## What changed between the two runs

Document length and formatting in the two caches, then the second run (N = 8, with the query vector) against the first run as generated and against the first run cut to 100 words, paired on the same queries.

In [5]:
for name, (queries, ids, qids, D, Q, base, E) in kept.items():
    per_query[name, "first run"] = testbed.score_run(hyde.search(hyde.hyde_vectors(E, Q), D, qids, ids), name)[1]
    for folder in ("generations", "generations_plain"):
        docs = [d for g in hyde.load_generations(name, folder).values() for d in g]
        words = sum(len(d.split()) for d in docs) / len(docs)
        marked = sum(bool(re.search(r"^\s*#|\*\*|^\s*Certainly", d, re.M)) for d in docs) / len(docs)
        print(f"{name} {folder:17} {words:4.0f} words per document, {marked:.0%} with a heading, bold mark or chat opening")
    for other in ("first run", "first 100 words", "cleaned, first 100 words"):
        c = testbed.compare(per_query[name, "second run"], per_query[name, other])
        print(f"{name} second run against {other:25} delta {c['delta']:+.3f} {c['ci95']} p={c['p']:.3g}")

scifact generations        364 words per document, 85% with a heading, bold mark or chat opening
scifact generations_plain   70 words per document, 0% with a heading, bold mark or chat opening
scifact second run against first run                 delta +0.074 [0.043, 0.1064] p=5.24e-06
scifact second run against first 100 words           delta -0.007 [-0.0295, 0.0145] p=0.523
scifact second run against cleaned, first 100 words  delta -0.008 [-0.0289, 0.0122] p=0.447


arguana generations        416 words per document, 88% with a heading, bold mark or chat opening


arguana generations_plain  116 words per document, 0% with a heading, bold mark or chat opening
arguana second run against first run                 delta +0.285 [0.2679, 0.3015] p=6.36e-175
arguana second run against first 100 words           delta +0.041 [0.0279, 0.0532] p=8.06e-10


arguana second run against cleaned, first 100 words  delta +0.027 [0.0149, 0.039] p=1.57e-05


## Concrete queries

On SciFact, the three queries where the second run gains most over plain Contriever and the three where it loses most, with the start of the first hypothetical document from each run.

In [6]:
queries, ids, qids, D, Q, base, _ = kept["scifact"]
first, second = hyde.load_generations("scifact"), hyde.load_generations("scifact", "generations_plain")
gain = per_query["scifact", "second run"] - base
order = gain.argsort()
for i in list(order[-3:][::-1]) + list(order[:3]):
    q = qids[i]
    print(f"{q}: {queries[q]}")
    print(f"  nDCG@10: Contriever {base[i]:.2f}, first run {per_query['scifact', 'first run'][i]:.2f}, second run {per_query['scifact', 'second run'][i]:.2f}")
    print(f"  first run:  {first_words(first[q][0], 40)}")
    print(f"  second run: {first_words(second[q][0], 40)}\n")

1121: Synaptic activity enhances local release of brain derived neurotrophic factor from postsynaptic dendrites.
  nDCG@10: Contriever 0.00, first run 1.00, second run 1.00
  first run:  **Passage: Supporting the Claim: Synaptic Activity Enhances Local Release of Brain-Derived Neurotrophic Factor (BDNF) from Postsynaptic Dendrites** Recent advancements in neuroscience have provided compelling evidence supporting the claim that synaptic activity enhances the local release of brain-derived neurotrophic factor (BDNF)
  second run: Recent studies have provided evidence supporting the claim that synaptic activity can enhance the local release of brain-derived neurotrophic factor (BDNF) from postsynaptic dendrites. Through in vivo and in vitro experiments, it was observed that during periods of heightened synaptic

1204: The combination of H3K4me3 and H3K79me2 is found in quiescent hair follicle stem cells.
  nDCG@10: Contriever 0.30, first run 0.43, second run 1.00
  first run:  ### Passage

## Leak check

Qwen may have seen the gold documents in training and could recite them. For every hypothetical document: the longest run of consecutive words it shares with a gold document of its query.

In [7]:
from difflib import SequenceMatcher

def shared(a, b):
    a, b = a.lower().split(), b.lower().split()
    m = SequenceMatcher(None, a, b, autojunk=False).find_longest_match(0, len(a), 0, len(b))
    return m.size, " ".join(a[m.a:m.a + m.size])

for folder in ("generations", "generations_plain"):
    for name in kept:
        corpus, queries, qrels = testbed.load(name)
        generations = hyde.load_generations(name, folder)
        best = sorted((max((shared(d, corpus[g]["text"]) for g in qrels[q] if g in corpus), default=(0, ""))
                       for q in queries for d in generations[q]), reverse=True)
        print(f"{name} {folder}: median {best[len(best) // 2][0]} words, longest {best[0][0]}, "
              f"{sum(s >= 8 for s, _ in best)} of {len(best)} documents share 8 words or more")
        print(f"  longest: {best[0][1][:150]}")

scifact generations: median 3 words, longest 10, 13 of 2400 documents share 8 words or more
  longest: was not associated with an increased risk of serious cardiovascular


arguana generations: median 3 words, longest 73, 26 of 11248 documents share 8 words or more
  longest: would in general be preferable to make such a change through normal legislation, but that was attempted in 1996 and found unconstitutional. supreme co
scifact generations_plain: median 2 words, longest 10, 4 of 2400 documents share 8 words or more
  longest: risk of serious cardiovascular events in young and middle-aged adults.


arguana generations_plain: median 2 words, longest 37, 15 of 11248 documents share 8 words or more
  longest: do not take these matters lightly in amending the constitution. however, i am convinced in this case it is the only way to provide the president with 


## Same encoder, two protocols

Everything above follows the paper: full documents, dot product on raw vectors. The testbed, and with it `method.py` and `results_variants/`, uses the first 200 words of each document and cosine similarity. Plain Contriever scores differently under the two, and the sign of HyDE's effect on SciFact differs too.

Here are the four combinations (documents full or 200 words, similarity dot product or cosine) on all three datasets: plain Contriever, and HyDE with N = 8 and the query vector for both caches, each HyDE line against plain Contriever under the same combination. For cosine every vector is normalised and the HyDE mean is renormalised, as in `method.py`. The (full, dot) lines repeat the results above; the (200 words, cosine) lines match `results_variants/contriever_hyde.json` within 0.002.

In [8]:
import numpy as np
unit = lambda X: X / np.linalg.norm(X, axis=-1, keepdims=True)

for name in ["scifact", "nfcorpus", "arguana"]:
    corpus, queries, qrels = testbed.load(name)
    ids, qids = list(corpus), list(queries)
    Ds = {"full": model.encode([(d["title"] + " " + d["text"]).strip() for d in corpus.values()], batch_size=32, convert_to_numpy=True),
          "200 words": model.encode(testbed.doc_texts(corpus), batch_size=32, convert_to_numpy=True)}
    Q = model.encode(list(queries.values()), batch_size=32, convert_to_numpy=True)
    Es = {"first run": hyde.embed_generations(model, queries, hyde.load_generations(name)),
          "second run": hyde.embed_generations(model, queries, hyde.load_generations(name, "generations_plain"))}
    for text, D in Ds.items():
        for sim in ("dot", "cosine"):
            D_, Q_ = (D, Q) if sim == "dot" else (unit(D), unit(Q))
            _, base, _ = testbed.score_run(hyde.search(Q_, D_, qids, ids), name)
            print(f"{name:8} {text:9} {sim:6} plain Contriever {base.mean():.4f}")
            for label, E in Es.items():
                V = hyde.hyde_vectors(E if sim == "dot" else unit(E), Q_, 8)
                _, nd, _ = testbed.score_run(hyde.search(V if sim == "dot" else unit(V), D_, qids, ids), name)
                c = testbed.compare(nd, base)
                print(f"{name:8} {text:9} {sim:6} HyDE {label:10} {nd.mean():.4f}  delta {c['delta']:+.3f} {c['ci95']} p={c['p']:.3g}")

scifact  full      dot    plain Contriever 0.6493
scifact  full      dot    HyDE first run  0.6075  delta -0.042 [-0.0765, -0.0073] p=0.0193


scifact  full      dot    HyDE second run 0.6820  delta +0.033 [0.0134, 0.0531] p=0.00173
scifact  full      cosine plain Contriever 0.5714


scifact  full      cosine HyDE first run  0.6857  delta +0.114 [0.0785, 0.1507] p=2.76e-09


scifact  full      cosine HyDE second run 0.6060  delta +0.035 [0.0111, 0.0586] p=0.00491


scifact  200 words dot    plain Contriever 0.6456
scifact  200 words dot    HyDE first run  0.6506  delta +0.005 [-0.025, 0.0353] p=0.746


scifact  200 words dot    HyDE second run 0.6760  delta +0.030 [0.0101, 0.0522] p=0.00572
scifact  200 words cosine plain Contriever 0.5963


... (14 outputs left out)

arguana  200 words dot    HyDE second run 0.4631  delta +0.048 [0.0295, 0.066] p=3.98e-07


arguana  200 words cosine plain Contriever 0.4675


arguana  200 words cosine HyDE first run  0.3750  delta -0.092 [-0.1122, -0.0731] p=5.8e-20


arguana  200 words cosine HyDE second run 0.4600  delta -0.007 [-0.0288, 0.0132] p=0.487


## What we conclude

- The reproduction as planned fails: with the paper's settings and Qwen2.5-3B, HyDE is below plain Contriever on both datasets (0.608 against 0.649 on SciFact, 0.192 against 0.379 on ArguAna; the paper reports 69.1 and 46.6).
- Under the paper's protocol (full documents, dot product) the cause is the length of the hypothetical documents. Removing the chat formatting changes nothing (0.607, 0.207). Keeping the first 100 words of the same documents turns the loss into a gain (0.689, 0.436). The second run, with 70 and 116 words per document and no formatting, beats Contriever on both datasets (0.682 and 0.477), in the direction and close to the size the paper reports.
- The protocol moves the baseline more than it moves HyDE. Plain Contriever scores 0.649 on SciFact with dot product and 0.571 with cosine on the same full documents (ArguAna: 0.379 and 0.450). Contriever is trained for dot product. Cutting documents to 200 words changes little on SciFact (0.646 with dot product) and helps on ArguAna (0.415).
- HyDE's gain follows the strength of the baseline it is measured against. First run on SciFact: -0.042 against the paper-protocol baseline, +0.005 with 200 words and dot product, +0.114 and +0.088 against the weakened cosine baselines. Second run on ArguAna: 0.460 to 0.477 in all four protocols, and the gain over plain Contriever shrinks from +0.098 to -0.007 as the baseline gets stronger. This is consistent with the pattern Weller et al. report across rankers.
- Length alone explains less than we first wrote. On ArguAna the first run loses in all four protocols (-0.187, -0.107, -0.116, -0.092), so long chat-style documents hurt there whatever the protocol. On SciFact they hurt only under dot product with full documents; under cosine the first run beats the second (0.686 against 0.606).
- Hypotheses, untested. ArguAna: a 400-word essay with an introduction, several points and a cut-off ending sits far from a short argument in embedding space. SciFact under dot product: the norm of a mean-pooled Contriever vector varies with the text, which matters for a dot product and disappears under cosine. The paper does not state either condition; `text-davinci-003` wrote short passages by default.
- The testbed rows use the testbed protocol (200 words, cosine) so that every ranker sees the same documents. `results_variants/contriever_hyde.json` is in that protocol. Both sets of Contriever numbers are reproduced above and differ only by protocol.
- Multiple comparisons: this notebook makes 68 comparisons (62 with Contriever, 6 between the two runs). With a Bonferroni threshold of 0.05 / 68 = 0.0007, these stand: the ArguAna first-run losses (p < 1e-19), the NFCorpus first-run losses under dot product (p < 1e-4), the SciFact first-run gains under cosine (p < 1e-6) and the ArguAna second-run gains with the query vector under dot product (p < 1e-6). These do not pass: the SciFact first-run loss under the paper's protocol (p = 0.02), the SciFact second-run gains (p = 0.002 to 0.07) and the ArguAna second-run lines under cosine. We call those consistent with the paper, not established.
- The checks, the second run and the protocol comparison were chosen after seeing test scores. They explain the failure; they are not a clean reproduction.
- Leak check: the median shared run is 2 to 3 words. Runs of 8 words or more occur in 17 of 4,800 SciFact documents and 41 of 22,496 ArguAna documents. The three longest ArguAna matches are sentences copied from the query passage, which shares them with its gold document; the longest SciFact ones (10 words) are built from the claim's own phrases.